# FYP Risk Analysis — Exploratory Notebook

This notebook demonstrates the ML risk prediction engine and lets you visualise results.

In [ ]:
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from app.services.risk_engine import predict_risk
from app.services.burndown_engine import compute_burndown
from app.services.grade_predictor import predict_grade
from app.database import execute_query

print('Environment ready!')

In [ ]:
# Load all projects
projects = pd.DataFrame(execute_query('SELECT * FROM projects'))
print(f'Loaded {len(projects)} projects')
projects.head()

In [ ]:
# Run risk prediction on all projects
risk_results = []
for _, row in projects.iterrows():
    try:
        r = predict_risk(int(row['id']))
        risk_results.append({'id': row['id'], 'title': row['title'], 'risk_score': r.risk_score, 'risk_level': r.risk_level, 'completion_probability': r.completion_probability})
    except Exception as e:
        print(f'Error for project {row["id"]}: {e}')

risk_df = pd.DataFrame(risk_results)
risk_df

In [ ]:
# Visualise risk distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk score histogram
axes[0].hist(risk_df['risk_score'], bins=10, color='coral', edgecolor='white')
axes[0].set_title('Risk Score Distribution')
axes[0].set_xlabel('Risk Score (0-100)')
axes[0].set_ylabel('Count')

# Risk level pie
level_counts = risk_df['risk_level'].value_counts()
colors = {'Low': '#22c55e', 'Medium': '#f59e0b', 'High': '#ef4444', 'Critical': '#7f1d1d'}
axes[1].pie(level_counts.values, labels=level_counts.index, colors=[colors.get(l, 'gray') for l in level_counts.index], autopct='%1.0f%%')
axes[1].set_title('Risk Level Breakdown')

plt.tight_layout()
plt.savefig('../data/risk_distribution.png', dpi=150)
plt.show()

In [ ]:
# Grade predictions
grade_results = []
for _, row in projects.iterrows():
    try:
        g = predict_grade(int(row['id']))
        grade_results.append({'id': row['id'], 'title': row['title'], 'grade': g.predicted_grade, 'score': g.predicted_score, 'confidence': g.confidence})
    except Exception as e:
        print(f'Error for project {row["id"]}: {e}')

grade_df = pd.DataFrame(grade_results)
print('Predicted Grade Distribution:')
print(grade_df['grade'].value_counts())
grade_df

In [ ]:
# Scatter: Risk Score vs Predicted Grade Score
if len(risk_df) > 0 and len(grade_df) > 0:
    merged = risk_df.merge(grade_df, on='id')
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(merged['risk_score'], merged['score'], c=merged['confidence'], cmap='RdYlGn', s=100, alpha=0.8)
    plt.colorbar(scatter, label='Confidence')
    plt.xlabel('Risk Score')
    plt.ylabel('Predicted Grade Score')
    plt.title('Risk vs Predicted Grade (colour = confidence)')
    for _, row in merged.iterrows():
        plt.annotate(row['title_x'][:15], (row['risk_score'], row['score']), fontsize=7, ha='right')
    plt.tight_layout()
    plt.savefig('../data/risk_vs_grade.png', dpi=150)
    plt.show()